In [23]:
import pandas as pd
import snappy
import ast

In [24]:
# We return any knot that has a lower bound equal to one and is alternating
df = pd.read_csv("knotinfo.csv")
df1 = df[(df["Unknotting Number"].str.startswith("[1")) & (df["Alternating"] == "Y")]
df1

,Name,Alternating,Unknotting Number
2981,13a_5,Y,[1;2]
2982,13a_6,Y,[1;2]
2983,13a_7,Y,[1;2]
2987,13a_11,Y,[1;2]
2988,13a_12,Y,[1;3]
...,...,...,...
7820,13a_4844,Y,[1;2]
7822,13a_4846,Y,[1;2]
7823,13a_4847,Y,[1;2]
7824,13a_4848,Y,[1;2]


In [29]:
#Notice, there are only three different unknotting number bounds given this dataset and the constraints
unique_bounds = df1["Unknotting Number"].unique()
for bound in unique_bounds:
    print(bound)

[1;2]
[1;3]
[1;4]


In [ ]:
unknotting_1 = 0
unknotting_2 = 0
bounds_improved = 0


def unknot_check(K):
    # We assume total_rank = 1 implies the knot K must be trivial
    K.simplify("global")
    if (int(K.knot_floer_homology()["total_rank"]) == 1):
        return True
    else:
        return False
    
def update_unknotting_val(index, found_unknot):
    global unknotting_1
    global unknotting_2
    global bounds_improved

    if found_unknot == True:
        df.loc[index, "Unknotting Number"] = "1"
        unknotting_1 += 1
    elif found_unknot == False:
        if df.loc[index, "Unknotting Number"] == "[1;2]":
            df.loc[index, "Unknotting Number"] = "2"
            unknotting_2 += 1   
        else:
            # We modify the lower bound to be 2
            new_bounds = list(row["Unknotting Number"])
            new_bounds[1] = "2"
            new_bounds = "".join(new_bounds)
            df.loc[index, "Unknotting Number"] = new_bounds
            bounds_improved += 1

            
for index, row in df1.iterrows():
    found_unknot = False
    i = 0

    # We convert the knotinfo name into a snappy Link name
    name = row["Name"]
    name = name.replace("_","")
    name = "K" + name

    # We convert the snappy Link name into a an interable DT code
    dt_code = list(snappy.Link(name).DT_code()[0])

    # We iterate through each possible crossing change until we find an unknot
    while i < len(dt_code) and found_unknot == False:
        new_dt = dt_code.copy()
        new_dt[i] = -1*new_dt[i] # We change a single crossing
        K = snappy.Link("DT: " + str(new_dt))
        if unknot_check(K) == True:
            found_unknot = True
        i += 1

    update_unknotting_val(index, found_unknot)

df.to_csv("updated_u(K)_values.csv", index=False)


print(f"Total Number of Knots Considered: {len(df1)}")
print(f"Number of Knots Determined to have Unknotting Number 1: {unknotting_1}")
print(f"Number of Knots Determined to have Unknotting Number 2: {unknotting_2}")
print(f"Number of Knots with Improved Unknotting Number Bounds: {bounds_improved}")
print("--------------------------------------------------------------------------------")    

     

